# Neural Network Classifier — Khumbu Icefall Early vs Late Era (PyTorch CNN, 128px)


In [1]:
import json, random, re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device, "| torch =", torch.__version__)

device = cuda | torch = 2.11.0+cu128


In [11]:
# Labels from filename year: Early (<=2014) vs Late (>=2015)
def parse_year(p):
    m = re.match(r"(\d{4})", Path(p).stem)
    return int(m.group(1)) if m else None

files = sorted(Path("dataset").glob("*.png"))
rows = [{"path": str(p), "year": parse_year(p)} for p in files]
df = pd.DataFrame(rows)
df["label"] = (df["year"] >= 2015).astype(int)  # 0=Early, 1=Late
print(df["label"].value_counts().rename({0: "Early", 1: "Late"}).to_string())

# Stratified 70/15/15 (seeded). Chronological split.csv is NOT used here:
# a time cutoff + time-ordered split would leave val/test single-class.
tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
splits = {"train": tr, "val": va, "test": te}
for k, v in splits.items():
    print(f"{k}: n={len(v)}  Early={(v['label']==0).sum()}  Late={(v['label']==1).sum()}")

df["split"] = "train"
df.loc[va.index, "split"] = "val"
df.loc[te.index, "split"] = "test"
Path("outputs").mkdir(exist_ok=True)
df.to_csv("outputs/nn_split.csv", index=False)
print("saved outputs/nn_split.csv")

label
Early    34
Late     27
train: n=42  Early=23  Late=19
val: n=9  Early=5  Late=4
test: n=10  Early=6  Late=4
saved outputs/nn_split.csv


In [3]:
# Transforms: train gets mild aug (rotation +-5, brightness/contrast +-10%);
# NO horizontal flip (week-2 rule: valley orientation / screenshot text).
# Eval transform is clean. Norm stats from week-2 train-only computation.
stats = json.load(open("outputs/eda_summary.json"))
MEAN, STD = stats["train_mean"], stats["train_std"]
IMG = 128

train_tfm = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tfm = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class KhumbuDataset(Dataset):
    def __init__(self, frame, tfm):
        self.frame = frame.reset_index(drop=True); self.tfm = tfm
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        return self.tfm(img), int(r["label"])

loaders = {
    "train": DataLoader(KhumbuDataset(splits["train"], train_tfm), batch_size=8, shuffle=True),
    "val": DataLoader(KhumbuDataset(splits["val"], eval_tfm), batch_size=8, shuffle=False),
    "test": DataLoader(KhumbuDataset(splits["test"], eval_tfm), batch_size=8, shuffle=False),
}
xb, yb = next(iter(loaders["train"]))
print("batch:", tuple(xb.shape), yb.tolist())

batch: (8, 3, 128, 128) [0, 1, 0, 0, 0, 1, 1, 0]


In [12]:
# CNN adapted from FashionMNISTCNN reference (Conv->ReLU->Pool x2,
# Flatten, Dropout, Linear) for 3-channel 128px input, 2-class output.
# AdaptiveAvgPool2d keeps the head small regardless of input size.
class CNN(nn.Module):
    def __init__(self, hidden=128, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        # 128->126->63->61->30 -> pooled to 64x4x4 = 1024
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64 * 4 * 4, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )
    def forward(self, x):
        return self.head(self.features(x))

n_params = sum(p.numel() for p in CNN().parameters())
print(f"params: {n_params:,}")

params: 150,850


In [13]:
def evaluate(model, loader):
    model.eval()  # dropout off
    y_pred, y_true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            y_pred += model(xb).argmax(dim=1).cpu().tolist()
            y_true += yb.tolist()
    acc = 100.0 * sum(p == t for p, t in zip(y_pred, y_true)) / len(y_true)
    return acc, y_pred, y_true

def train_config(hidden, dropout, lr, wd, max_epochs=25, patience=6):
    model = CNN(hidden=hidden, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    counts = splits["train"]["label"].value_counts().sort_index().values.astype(float)
    w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    hist = {"tr_loss": [], "va_f1": []}
    best = {"f1": -1.0, "state": None}
    bad = 0
    for epoch in range(max_epochs):
        model.train()
        tot = 0.0
        for xb, yb in loaders["train"]:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(xb)
        hist["tr_loss"].append(tot / len(splits["train"]))
        _, yp, yt = evaluate(model, loaders["val"])
        f1 = f1_score(yt, yp, average="macro", zero_division=0)
        hist["va_f1"].append(f1)
        if f1 > best["f1"]:
            best = {"f1": f1, "state": {k: v.cpu() for k, v in model.state_dict().items()}}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})
    return model, hist, best["f1"]

In [6]:
# Hyperparameter grid: lr x hidden x dropout (wd fixed at 1e-4).
# Observe: higher lr trains faster but oscillates; larger dropout
# regularizes the tiny train set; larger hidden adds capacity/overfit risk.
import itertools
GRID = list(itertools.product([1e-3, 1e-4], [128, 256], [0.3, 0.5]))
results, states, hists = [], {}, {}
for lr, hidden, drop in GRID:
    m, h, f1 = train_config(hidden=hidden, dropout=drop, lr=lr, wd=1e-4)
    key = f"lr={lr} hidden={hidden} drop={drop}"
    results.append({"config": key, "lr": lr, "hidden": hidden,
                    "dropout": drop, "val_macro_f1": round(f1, 4)})
    states[key] = {k: v.cpu() for k, v in m.state_dict().items()}
    hists[key] = h
    print(f"{key}: val_macro_f1={f1:.4f}")

res_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
print(res_df.to_string(index=False))
BEST_KEY = res_df.iloc[0]["config"]
print("BEST:", BEST_KEY)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for key, h in hists.items():
    lw = 2.5 if key == BEST_KEY else 1.0
    ax[0].plot(h["tr_loss"], linewidth=lw, label=key)
    ax[1].plot(h["va_f1"], linewidth=lw, label=key)
ax[0].set_title("Train loss per config"); ax[0].set_xlabel("epoch")
ax[1].set_title("Val macro-F1 per config"); ax[1].set_xlabel("epoch")
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig("outputs/figures/09_hparam_curves.png", dpi=120)
print("saved outputs/figures/09_hparam_curves.png")

lr=0.001 hidden=128 drop=0.3: val_macro_f1=0.7750
lr=0.001 hidden=128 drop=0.5: val_macro_f1=0.7750
lr=0.001 hidden=256 drop=0.3: val_macro_f1=1.0000
lr=0.001 hidden=256 drop=0.5: val_macro_f1=1.0000
lr=0.0001 hidden=128 drop=0.3: val_macro_f1=0.8831
lr=0.0001 hidden=128 drop=0.5: val_macro_f1=0.7750
lr=0.0001 hidden=256 drop=0.3: val_macro_f1=0.7750
lr=0.0001 hidden=256 drop=0.5: val_macro_f1=0.7750
                       config     lr  hidden  dropout  val_macro_f1
 lr=0.001 hidden=256 drop=0.5 0.0010     256      0.5        1.0000
 lr=0.001 hidden=256 drop=0.3 0.0010     256      0.3        1.0000
lr=0.0001 hidden=128 drop=0.3 0.0001     128      0.3        0.8831
 lr=0.001 hidden=128 drop=0.5 0.0010     128      0.5        0.7750
 lr=0.001 hidden=128 drop=0.3 0.0010     128      0.3        0.7750
lr=0.0001 hidden=128 drop=0.5 0.0001     128      0.5        0.7750
lr=0.0001 hidden=256 drop=0.3 0.0001     256      0.3        0.7750
lr=0.0001 hidden=256 drop=0.5 0.0001     256      0.

In [7]:
# Final test evaluation (run once, best-on-val config only).
names = ["Early", "Late"]
best = CNN(hidden=int(res_df.iloc[0]["hidden"]),
             dropout=float(res_df.iloc[0]["dropout"])).to(device)
r = res_df.iloc[0]
best.load_state_dict({k: v.to(device) for k, v in states[r['config']].items()})
test_acc, y_pred, y_true = evaluate(best, loaders["test"])
print(f"test acc: {test_acc:.2f}%")
report = classification_report(y_true, y_pred, target_names=names,
                               digits=4, zero_division=0)
print(report)

disp = ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, labels=[0, 1], display_labels=names, cmap="Blues")
plt.title(f"Khumbu Early vs Late (best: {BEST_KEY})")
plt.savefig("outputs/figures/08_confusion_matrix.png", dpi=120, bbox_inches="tight")
print("saved outputs/figures/08_confusion_matrix.png")

payload = {
    "best_config": {"lr": float(r["lr"]), "hidden": int(r["hidden"]),
                      "dropout": float(r["dropout"]), "wd": 1e-4},
    "grid": results,
    "test_accuracy": round(test_acc, 2),
    "classification_report": report,
    "confusion": {"y_true": y_true, "y_pred": y_pred},
}
json.dump(payload, open("outputs/nn_results.json", "w"), indent=2)
torch.save(best.state_dict(), "outputs/khumbu_cnn.pt")
print("saved outputs/nn_results.json + outputs/khumbu_cnn.pt")

test acc: 80.00%
              precision    recall  f1-score   support

       Early     0.7500    1.0000    0.8571         6
        Late     1.0000    0.5000    0.6667         4

    accuracy                         0.8000        10
   macro avg     0.8750    0.7500    0.7619        10
weighted avg     0.8500    0.8000    0.7810        10

saved outputs/figures/08_confusion_matrix.png
saved outputs/nn_results.json + outputs/khumbu_cnn.pt


In [8]:
# Baseline: sklearn MLP on 64x64 grayscale vectors (zero-install reference point).
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

def flat_vec(p):
    return np.asarray(Image.open(p).convert("L").resize((64, 64)),
                      dtype=np.float32).ravel() / 255.0

Xtr = np.stack([flat_vec(p) for p in splits["train"]["path"]])
ytr = splits["train"]["label"].values
Xte = np.stack([flat_vec(p) for p in splits["test"]["path"]])
yte = splits["test"]["label"].values
sc = StandardScaler()
mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=800, random_state=SEED)
mlp.fit(sc.fit_transform(Xtr), ytr)
yp = mlp.predict(sc.transform(Xte))
print("MLP baseline test acc: %.2f%%" % (100 * (yp == yte).mean()))
print(classification_report(yte, yp, target_names=["Early", "Late"],
                            digits=4, zero_division=0))

MLP baseline test acc: 100.00%
              precision    recall  f1-score   support

       Early     1.0000    1.0000    1.0000         6
        Late     1.0000    1.0000    1.0000         4

    accuracy                         1.0000        10
   macro avg     1.0000    1.0000    1.0000        10
weighted avg     1.0000    1.0000    1.0000        10



In [9]:
# Reload check: weights saved above must reproduce the same test accuracy.
r = json.load(open("outputs/nn_results.json"))["best_config"]
m2 = CNN(hidden=r["hidden"], dropout=r["dropout"]).to(device)
m2.load_state_dict(torch.load("outputs/khumbu_cnn.pt", map_location=device,
                              weights_only=True))
acc2, _, _ = evaluate(m2, loaders["test"])
print(f"reloaded test acc: {acc2:.2f}%")

reloaded test acc: 80.00%


## Visualization — CNN Training & Evaluation Charts
Six charts summarising the dataset, hyperparameter search, and final model performance.

In [10]:
# ── Visualization cell ──────────────────────────────────────────────────────
# Requires: res_df, hists, BEST_KEY, y_true, y_pred, splits, df (defined above)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from matplotlib.colors import LinearSegmentedColormap

Path("outputs/figures").mkdir(parents=True, exist_ok=True)

# ── shared dark-theme style ──────────────────────────────────────────────────
PALETTE  = ["#5B8FF9", "#FF7875"]   # blue=Early  red=Late
DARK_BG  = "#1a1a2e"
CARD_BG  = "#16213e"
TEXT_CLR = "#e0e0e0"
GRID_CLR = "#2a2a4a"
plt.rcParams.update({
    "figure.facecolor": DARK_BG,
    "axes.facecolor":   CARD_BG,
    "axes.edgecolor":   GRID_CLR,
    "axes.labelcolor":  TEXT_CLR,
    "axes.titlecolor":  TEXT_CLR,
    "xtick.color":      TEXT_CLR,
    "ytick.color":      TEXT_CLR,
    "text.color":       TEXT_CLR,
    "grid.color":       GRID_CLR,
    "grid.linewidth":   0.5,
    "font.size":        11,
})

# ── CHART 1 — Class distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4.5))
cls_labels = ["Early (<=2014)", "Late (>=2015)"]
cls_counts = [(df["label"] == 0).sum(), (df["label"] == 1).sum()]
bars = ax.bar(cls_labels, cls_counts, color=PALETTE, width=0.5, edgecolor="none", zorder=3)
ax.set_title("Dataset Class Distribution", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Image Count")
ax.yaxis.grid(True, zorder=0); ax.set_axisbelow(True)
for bar, v in zip(bars, cls_counts):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, str(v),
            ha="center", va="bottom", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figures/10_class_distribution.png", dpi=130, bbox_inches="tight")
plt.show(); print("saved 10_class_distribution.png")

# ── CHART 2 — Train / Val / Test split breakdown (stacked bar) ───────────────
spl_names = ["Train", "Val", "Test"]
e_counts  = [(splits[s]["label"] == 0).sum() for s in ["train", "val", "test"]]
l_counts  = [(splits[s]["label"] == 1).sum() for s in ["train", "val", "test"]]
x = np.arange(len(spl_names))
fig, ax = plt.subplots(figsize=(6, 4.5))
b1 = ax.bar(x, e_counts, color=PALETTE[0], label="Early", width=0.45, edgecolor="none", zorder=3)
b2 = ax.bar(x, l_counts, color=PALETTE[1], label="Late",  width=0.45,
            bottom=e_counts, edgecolor="none", zorder=3)
for be, bl, ec, lc in zip(b1, b2, e_counts, l_counts):
    ax.text(be.get_x()+be.get_width()/2, ec/2,    str(ec),
            ha="center", va="center", fontsize=11, fontweight="bold", color="white")
    ax.text(bl.get_x()+bl.get_width()/2, ec+lc/2, str(lc),
            ha="center", va="center", fontsize=11, fontweight="bold", color="white")
ax.set_xticks(x); ax.set_xticklabels(spl_names)
ax.set_title("Dataset Split — Early vs Late per Subset", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Image Count")
ax.yaxis.grid(True, zorder=0); ax.set_axisbelow(True)
ax.legend(framealpha=0.2, edgecolor=GRID_CLR)
plt.tight_layout()
plt.savefig("outputs/figures/11_split_breakdown.png", dpi=130, bbox_inches="tight")
plt.show(); print("saved 11_split_breakdown.png")

# ── CHART 3 — Hyperparameter val macro-F1 heatmap (lr x hidden per dropout) ─
nn_res   = json.load(open("outputs/nn_results.json"))
grid_res = nn_res["grid"]
lrs      = sorted(set(g["lr"]      for g in grid_res))
hiddens  = sorted(set(g["hidden"]  for g in grid_res))
dropouts = sorted(set(g["dropout"] for g in grid_res))
cmap_heat = LinearSegmentedColormap.from_list("khumbu", [DARK_BG, "#5B8FF9", "#a8ff78"], N=256)
fig, axes = plt.subplots(1, len(dropouts), figsize=(10, 4), sharey=True)
fig.suptitle("Val Macro-F1 Heatmap (lr x hidden, per dropout)",
             fontsize=13, fontweight="bold", y=1.02)
for ax_h, drop in zip(axes, dropouts):
    mat = np.zeros((len(lrs), len(hiddens)))
    for g in grid_res:
        if g["dropout"] == drop:
            mat[lrs.index(g["lr"]), hiddens.index(g["hidden"])] = g["val_macro_f1"]
    im = ax_h.imshow(mat, cmap=cmap_heat, vmin=0.7, vmax=1.0, aspect="auto")
    ax_h.set_xticks(range(len(hiddens))); ax_h.set_xticklabels([str(h) for h in hiddens])
    ax_h.set_yticks(range(len(lrs)));    ax_h.set_yticklabels([str(l) for l in lrs])
    ax_h.set_xlabel("Hidden size"); ax_h.set_title(f"dropout={drop}", fontsize=11)
    if ax_h is axes[0]: ax_h.set_ylabel("Learning rate")
    for ri in range(len(lrs)):
        for ci in range(len(hiddens)):
            v = mat[ri, ci]
            ax_h.text(ci, ri, f"{v:.3f}", ha="center", va="center",
                      fontsize=10, fontweight="bold",
                      color="black" if v > 0.9 else TEXT_CLR)
fig.colorbar(im, ax=axes[-1], label="Val Macro-F1", shrink=0.85)
plt.tight_layout()
plt.savefig("outputs/figures/12_hparam_heatmap.png", dpi=130, bbox_inches="tight")
plt.show(); print("saved 12_hparam_heatmap.png")

# ── CHART 4 — Per-config val macro-F1 horizontal bar ────────────────────────
sorted_grid = sorted(grid_res, key=lambda g: g["val_macro_f1"])
cfg_labels  = [g["config"]       for g in sorted_grid]
cfg_f1      = [g["val_macro_f1"] for g in sorted_grid]
bar_colors  = ["#a8ff78" if g["config"] == BEST_KEY else "#5B8FF9" for g in sorted_grid]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(cfg_labels, cfg_f1, color=bar_colors, edgecolor="none", height=0.55, zorder=3)
ax.set_xlim(0.6, 1.10)
ax.xaxis.grid(True, zorder=0); ax.set_axisbelow(True)
ax.set_xlabel("Val Macro-F1")
ax.set_title("Hyperparameter Configs — Val Macro-F1", fontsize=13, fontweight="bold", pad=12)
for bar, v in zip(bars, cfg_f1):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
            f"{v:.4f}", va="center", fontsize=10)
ax.legend(handles=[
    mpatches.Patch(color="#a8ff78", label="Best config"),
    mpatches.Patch(color="#5B8FF9", label="Other configs"),
], framealpha=0.2, edgecolor=GRID_CLR)
plt.tight_layout()
plt.savefig("outputs/figures/13_config_f1_bar.png", dpi=130, bbox_inches="tight")
plt.show(); print("saved 13_config_f1_bar.png")

# ── CHART 5 — Confusion matrix (styled) ─────────────────────────────────────
cm_data = nn_res["confusion"]
yt, yp  = cm_data["y_true"], cm_data["y_pred"]
cm_mat  = confusion_matrix(yt, yp, labels=[0, 1])
cmap_cm = LinearSegmentedColormap.from_list("cm_blue", [CARD_BG, "#5B8FF9", "#a8ff78"], N=256)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_mat, cmap=cmap_cm, aspect="auto")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Early", "Late"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Early", "Late"])
ax.set_xlabel("Predicted Label"); ax.set_ylabel("True Label")
ax.set_title(f"Confusion Matrix — CNN Test Set\nBest: {BEST_KEY}",
             fontsize=11, fontweight="bold", pad=10)
for i in range(2):
    for j in range(2):
        v = cm_mat[i, j]
        ax.text(j, i, str(v), ha="center", va="center", fontsize=22, fontweight="bold",
                color="black" if v > cm_mat.max() * 0.6 else TEXT_CLR)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig("outputs/figures/14_confusion_matrix_styled.png", dpi=130, bbox_inches="tight")
plt.show(); print("saved 14_confusion_matrix_styled.png")

# ── CHART 6 — Per-class precision / recall / F1 ─────────────────────────────
prec, rec, f1s, _ = precision_recall_fscore_support(yt, yp, labels=[0, 1], zero_division=0)
metrics    = ["Precision", "Recall", "F1-Score"]
early_vals = [prec[0], rec[0], f1s[0]]
late_vals  = [prec[1], rec[1], f1s[1]]
x_m   = np.arange(len(metrics))
width = 0.32
fig, ax = plt.subplots(figsize=(7, 4.5))
be = ax.bar(x_m - width/2, early_vals, width, color=PALETTE[0], label="Early",
            edgecolor="none", zorder=3)
bl = ax.bar(x_m + width/2, late_vals,  width, color=PALETTE[1], label="Late",
            edgecolor="none", zorder=3)
ax.set_xticks(x_m); ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.15)
ax.yaxis.grid(True, zorder=0); ax.set_axisbelow(True)
ax.set_ylabel("Score")
ax.set_title("Per-Class Metrics — CNN Test Set", fontsize=13, fontweight="bold", pad=12)
for bar in list(be) + list(bl):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02,
            f"{h:.2f}", ha="center", va="bottom", fontsize=10)
ax.legend(framealpha=0.2, edgecolor=GRID_CLR)
plt.tight_layout()
plt.savefig("outputs/figures/15_per_class_metrics.png", dpi=130, bbox_inches="tight")
plt.show(); print("saved 15_per_class_metrics.png")

print("\n✓ All 6 visualization charts saved to outputs/figures/")


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17800\2545533451.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print("saved 10_class_distribution.png")


saved 10_class_distribution.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17800\2545533451.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print("saved 11_split_breakdown.png")


saved 11_split_breakdown.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17800\2545533451.py:100: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print("saved 12_hparam_heatmap.png")


saved 12_hparam_heatmap.png


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17800\2545533451.py:122: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print("saved 13_config_f1_bar.png")
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17800\2545533451.py:144: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print("saved 14_confusion_matrix_styled.png")


saved 13_config_f1_bar.png
saved 14_confusion_matrix_styled.png
saved 15_per_class_metrics.png

✓ All 6 visualization charts saved to outputs/figures/


C:\Users\ASUS\AppData\Local\Temp\ipykernel_17800\2545533451.py:170: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print("saved 15_per_class_metrics.png")
